In [ ]:
import os, sys, urllib.request, warnings, subprocess
warnings.filterwarnings('ignore')

for pkg in ['optuna', 'albumentations', 'gdown', 'openpyxl', 'h5py']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import cv2
import glob, random, h5py, json
from tqdm import tqdm
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── custom_losses 경로 (rules.md §6-1) ───────────────────────────────────────
sys.path.insert(0, '/root/imbalanced-data-LWCE/medical_data')
from custom_losses import get_loss_function, calculate_weights

# ── Google Drive 마운트 (Colab) ───────────────────────────────────────────────
# Synapse 데이터 Drive 업로드 경로:
#   MyDrive/imbalanced-data-LWCE/synapse/
#     train_npz/*.npz
#     test_vol_h5/*.npy.h5
GDRIVE_DATA_PATH = '/content/drive/MyDrive/imbalanced-data-LWCE/synapse'

IS_COLAB = False
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IS_COLAB = True
    print('Google Drive 마운트 완료')
except Exception:
    print('Colab 환경 아님 — 로컬/수동 경로 사용')

# ── 결과 저장 경로 ────────────────────────────────────────────────────────────
RESULTS_DIR = '/root/imbalanced-data-LWCE/medical_data/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── TransUNet 클론 및 ViT 가중치 ──────────────────────────────────────────────
TRANSUNET_DIR = '/tmp/TransUNet'
if not os.path.exists(TRANSUNET_DIR):
    print('TransUNet 클론 중...')
    os.system(f'git clone https://github.com/Beckschen/TransUNet.git {TRANSUNET_DIR}')

for p in [TRANSUNET_DIR, os.path.join(TRANSUNET_DIR, 'networks')]:
    if p not in sys.path:
        sys.path.insert(0, p)

PRETRAINED_DIR = os.path.join(TRANSUNET_DIR, 'model/vit_checkpoint/imagenet21k')
os.makedirs(PRETRAINED_DIR, exist_ok=True)
VIT_WEIGHTS = os.path.join(PRETRAINED_DIR, 'R50+ViT-B_16.npz')
if not os.path.exists(VIT_WEIGHTS):
    print('ViT-R50+B/16 가중치 다운로드 중...')
    url = 'https://storage.googleapis.com/vit_models/imagenet21k/R50+ViT-B_16.npz'
    urllib.request.urlretrieve(url, VIT_WEIGHTS)
    print('다운로드 완료')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print('환경 설정 완료')

In [ ]:
# ── Cell 1: Synapse 데이터 로드 ───────────────────────────────────────────────
#
# [데이터 준비 방법]
#   1. TransUNet 공식 전처리 데이터 다운로드:
#      https://drive.google.com/drive/folders/1ACJEoTp-uqfFJ73qS3eUObQh52nGuzCd
#   2. 압축 해제 후 구글 드라이브에 업로드:
#      MyDrive/imbalanced-data-LWCE/synapse/train_npz/  (*.npz 슬라이스)
#      MyDrive/imbalanced-data-LWCE/synapse/test_vol_h5/ (*.npy.h5 볼륨)
# ─────────────────────────────────────────────────────────────────────────────

DATA_DIR = '/tmp/synapse_data'
os.makedirs(DATA_DIR, exist_ok=True)

# ── Google Drive → /tmp 복사 ──────────────────────────────────────────────────
if IS_COLAB and os.path.exists(GDRIVE_DATA_PATH):
    import shutil
    if not os.path.exists(os.path.join(DATA_DIR, 'train_npz')):
        print('Google Drive에서 Synapse 데이터 복사 중...')
        shutil.copytree(GDRIVE_DATA_PATH, DATA_DIR, dirs_exist_ok=True)
        print('복사 완료')
    else:
        print('캐시 사용: /tmp/synapse_data 이미 존재')
else:
    # 로컬 환경 또는 Drive 미마운트: 수동 배치 확인
    if not os.path.exists(os.path.join(DATA_DIR, 'train_npz')):
        print('[데이터 없음] 아래 중 하나를 선택하세요:')
        print('  옵션 A (Colab): Cell 0의 GDRIVE_DATA_PATH 확인 후 재실행')
        print('  옵션 B (로컬):  /tmp/synapse_data/train_npz/ 에 .npz 파일 직접 배치')
        print('  다운로드: https://drive.google.com/drive/folders/1ACJEoTp-uqfFJ73qS3eUObQh52nGuzCd')

TRAIN_NPZ_DIR = os.path.join(DATA_DIR, 'train_npz')
TEST_H5_DIR   = os.path.join(DATA_DIR, 'test_vol_h5')

train_files = sorted(glob.glob(os.path.join(TRAIN_NPZ_DIR, '*.npz')))
test_files  = sorted(glob.glob(os.path.join(TEST_H5_DIR,   '*.npy.h5')))
print(f'Train slices: {len(train_files)}  |  Test volumes: {len(test_files)}')
assert len(train_files) > 0, 'Train 데이터 없음. 위 안내를 따라 데이터를 배치하세요.'

# ── 클래스 정의 ───────────────────────────────────────────────────────────────
NUM_CLASSES = 9
CLASS_NAMES = ['background', 'aorta', 'gallbladder', 'spleen',
               'left_kidney', 'right_kidney', 'liver', 'stomach', 'pancreas']

# ── Dataset 클래스 ────────────────────────────────────────────────────────────
class SynapseDataset(Dataset):
    def __init__(self, npz_files, img_size=224, augment=False):
        self.files    = npz_files
        self.img_size = img_size
        self.augment  = augment

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        data  = np.load(self.files[idx])
        image = data['image'].astype(np.float32)
        label = data['label'].astype(np.int64)

        image = cv2.resize(image, (self.img_size, self.img_size), interpolation=cv2.INTER_LINEAR)
        label = cv2.resize(label, (self.img_size, self.img_size), interpolation=cv2.INTER_NEAREST)

        image = (image - image.min()) / (image.max() - image.min() + 1e-8)
        image = np.stack([image] * 3, axis=0).astype(np.float32)

        if self.augment:
            img_hw = image.transpose(1, 2, 0)
            if random.random() > 0.5: img_hw = np.fliplr(img_hw).copy(); label = np.fliplr(label).copy()
            if random.random() > 0.5: img_hw = np.flipud(img_hw).copy(); label = np.flipud(label).copy()
            image = img_hw.transpose(2, 0, 1)

        return torch.from_numpy(image), torch.from_numpy(label).long()

# ── DataLoader ────────────────────────────────────────────────────────────────
tr_files, val_files = train_test_split(train_files, test_size=0.1, random_state=42)
train_loader = DataLoader(SynapseDataset(tr_files,  augment=True),  batch_size=12, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(SynapseDataset(val_files, augment=False), batch_size=12, shuffle=False, num_workers=4, pin_memory=True)
print(f'Train: {len(tr_files)} slices  |  Val: {len(val_files)} slices')

# ── 클래스 비율 계산 ──────────────────────────────────────────────────────────
print('클래스 비율 계산 중...')
class_counts = np.zeros(NUM_CLASSES, dtype=np.int64)
for fp in tqdm(tr_files, desc='Counting'):
    lbl = np.load(fp)['label'].astype(np.int64)
    for c in range(NUM_CLASSES):
        class_counts[c] += int((lbl == c).sum())
class_counts = class_counts.tolist()
print('\nClass pixel counts:')
for i, (n, c) in enumerate(zip(CLASS_NAMES, class_counts)):
    ratio = class_counts[0] / (c + 1)
    print(f'  [{i}] {n:<15}: {c:>12,}  (BG:FG = {ratio:.0f}:1)')

In [ ]:
# ── TransUNet 모델 로드 ───────────────────────────────────────────────────────
from vit_seg_modeling import VisionTransformer as ViT_seg
from vit_seg_modeling import CONFIGS as CONFIGS_ViT_seg

config_vit = CONFIGS_ViT_seg['R50-ViT-B_16']
config_vit.n_classes   = NUM_CLASSES
config_vit.n_skip       = 3
config_vit.patches.grid = (14, 14)   # 224 / 16

def build_transunet():
    model = ViT_seg(config_vit, img_size=224, num_classes=NUM_CLASSES)
    model.load_from(weights=np.load(VIT_WEIGHTS))
    return model.to(device)

# ── 검증 함수 ─────────────────────────────────────────────────────────────────
def compute_val_metrics(model, loader):
    """Val Dice (클래스별 + mDice) 계산"""
    model.eval()
    dice_per_class = np.zeros(NUM_CLASSES - 1)  # background 제외
    n_batches = 0

    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device)
            logits = model(imgs)
            preds  = torch.argmax(logits, dim=1)

            for c_idx, c in enumerate(range(1, NUM_CLASSES)):  # fg classes only
                p = (preds  == c).float()
                t = (masks  == c).float()
                inter = (p * t).sum()
                union = p.sum() + t.sum()
                if union > 0:
                    dice_per_class[c_idx] += (2. * inter / (union + 1e-8)).item()

            n_batches += 1

    dice_per_class /= n_batches
    return dice_per_class, float(np.mean(dice_per_class))

print("TransUNet 모델 준비 완료 (build_transunet() 로 인스턴스 생성)")
print(f"클래스 수: {NUM_CLASSES}  |  Foreground classes: {CLASS_NAMES[1:]}")


In [ ]:
def train_transunet(loss_name, alpha=1.0, epochs=30, lr=1e-4,
                    subset_ratio=1.0, tag=""):
    """
    loss_name    : 'ce_dice' | 'plwce_dice' | 'pwce_dice'
    alpha        : PLWCE / PWCE의 alpha 파라미터
    subset_ratio : Optuna 탐색용 축소 비율 (0~1). 1.0이면 전체 데이터
    tag          : 식별 태그 (저장 파일명용)
    """
    model     = build_transunet()
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    # ── CE 부분 교체 핵심 ──────────────────────────────────────────────────────
    # 기존: ce_loss = CrossEntropyLoss()  +  dice_loss = DiceLoss()
    # 변경: get_loss_function() 한 줄로 CE + Dice 통합 (alpha 파라미터 적용)
    criterion = get_loss_function(loss_name, class_counts=class_counts, alpha=alpha)
    # ─────────────────────────────────────────────────────────────────────────

    name = f"{loss_name}_alpha{alpha:.2f}" if alpha != 1.0 else loss_name
    if tag: name = f"{tag}_{name}"
    print(f"\n{'='*60}\n{name}  (epochs={epochs}, subset={subset_ratio:.0%})\n{'='*60}")

    # subset_ratio 적용 (Optuna 탐색 시 빠른 평가용)
    if subset_ratio < 1.0:
        n = max(1, int(len(train_loader.dataset) * subset_ratio))
        sub_ds    = torch.utils.data.Subset(train_loader.dataset,
                                            random.sample(range(len(train_loader.dataset)), n))
        sub_loader = DataLoader(sub_ds, batch_size=12, shuffle=True, num_workers=2)
    else:
        sub_loader = train_loader

    best_dice = 0.0
    history   = {'loss': [], 'val_mdice': []}

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0

        for imgs, masks in tqdm(sub_loader, desc=f"Ep{epoch+1:02d}/{epochs}", leave=False):
            imgs, masks = imgs.to(device), masks.to(device)
            optimizer.zero_grad()

            logits = model(imgs)                    # (B, 9, H, W)
            loss   = criterion(logits, masks)       # CE 교체 부분
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        avg_loss = epoch_loss / len(sub_loader)
        _, mdice = compute_val_metrics(model, val_loader)
        history['loss'].append(avg_loss)
        history['val_mdice'].append(mdice)

        print(f"Ep{epoch+1:02d} | Loss:{avg_loss:.4f} | Val mDice:{mdice:.4f}", end="")
        if mdice > best_dice:
            best_dice = mdice
            torch.save(model.state_dict(), f'/tmp/best_transunet_{name}.pth')
            print("  <- Best!", end="")
        print()

    model.load_state_dict(torch.load(f'/tmp/best_transunet_{name}.pth', weights_only=True))
    print(f"최고 Val mDice: {best_dice:.4f}")
    return model, history, best_dice

print("train_transunet() 함수 정의 완료")


In [ ]:
# ── Optuna objective 함수 ────────────────────────────────────────────────────
# tabular_data/scr/optuna_tuner_alpha_only.py 패턴 참조:
#   - trial.suggest_float('alpha', low, high)
#   - MedianPruner로 나쁜 trial 조기 종료
#   - 축소 데이터(subset_ratio=0.15) + 5 epochs 로 빠른 proxy 평가

ALPHA_LOW  = 2.5
ALPHA_HIGH = 15.0
PROXY_EPOCHS     = 5     # 탐색용 빠른 훈련
PROXY_SUBSET     = 0.15  # 전체 슬라이스의 15%만 사용
N_TRIALS         = 30    # 탐색 횟수

def make_alpha_objective(loss_name):
    """loss_name 별 Optuna objective 생성"""
    def objective(trial):
        alpha = trial.suggest_float('alpha', ALPHA_LOW, ALPHA_HIGH)
        try:
            _, _, mdice = train_transunet(
                loss_name   = loss_name,
                alpha       = alpha,
                epochs      = PROXY_EPOCHS,
                subset_ratio= PROXY_SUBSET,
                tag         = f"trial{trial.number}"
            )
            return mdice
        except Exception as e:
            print(f"Trial {trial.number} 실패: {e}")
            return 0.0
    return objective

# ── PLWCE alpha 탐색 ──────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"[Optuna] PLWCE alpha 탐색  (범위: {ALPHA_LOW}~{ALPHA_HIGH},  {N_TRIALS} trials)")
print(f"{'='*60}")

study_plwce = optuna.create_study(
    direction  = 'maximize',
    study_name = 'transunet_plwce_alpha',
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)
study_plwce.optimize(make_alpha_objective('plwce_dice'),
                     n_trials=N_TRIALS, show_progress_bar=True)

best_alpha_plwce = study_plwce.best_params['alpha']
print(f"\n[PLWCE] 최적 alpha = {best_alpha_plwce:.4f}  (Val mDice = {study_plwce.best_value:.4f})")

# ── PWCE alpha 탐색 ───────────────────────────────────────────────────────────
print(f"\n{'='*60}")
print(f"[Optuna] PWCE alpha 탐색  (범위: {ALPHA_LOW}~{ALPHA_HIGH},  {N_TRIALS} trials)")
print(f"{'='*60}")

study_pwce = optuna.create_study(
    direction  = 'maximize',
    study_name = 'transunet_pwce_alpha',
    pruner     = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=2)
)
study_pwce.optimize(make_alpha_objective('pwce_dice'),
                    n_trials=N_TRIALS, show_progress_bar=True)

best_alpha_pwce = study_pwce.best_params['alpha']
print(f"\n[PWCE] 최적 alpha = {best_alpha_pwce:.4f}  (Val mDice = {study_pwce.best_value:.4f})")

# ── Optuna 결과 저장 ──────────────────────────────────────────────────────────
optuna_results = {
    'plwce': {'best_alpha': best_alpha_plwce, 'best_proxy_mdice': study_plwce.best_value,
              'trials': [{'number': t.number, 'alpha': t.params.get('alpha'),
                          'value': t.value} for t in study_plwce.trials if t.value]},
    'pwce':  {'best_alpha': best_alpha_pwce,  'best_proxy_mdice': study_pwce.best_value,
              'trials': [{'number': t.number, 'alpha': t.params.get('alpha'),
                          'value': t.value} for t in study_pwce.trials if t.value]},
}
with open(os.path.join(RESULTS_DIR, 'pancreas_optuna_results.json'), 'w') as f:
    json.dump(optuna_results, f, indent=2, ensure_ascii=False)
print(f"\nOptuna 결과 저장: {os.path.join(RESULTS_DIR, 'pancreas_optuna_results.json')}")

# ── Optuna 탐색 곡선 시각화 ───────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, study, name in [(axes[0], study_plwce, 'PLWCE'), (axes[1], study_pwce, 'PWCE')]:
    trials   = [t for t in study.trials if t.value is not None]
    alphas   = [t.params['alpha'] for t in trials]
    values   = [t.value for t in trials]
    best_a   = study.best_params['alpha']
    best_v   = study.best_value

    ax.scatter(alphas, values, alpha=0.5, s=40, label='Trials')
    ax.axvline(best_a, color='red', linestyle='--', label=f'Best α={best_a:.2f}')
    ax.scatter([best_a], [best_v], color='red', s=100, zorder=5)
    ax.set_xlabel('alpha'); ax.set_ylabel('Val mDice (proxy)')
    ax.set_title(f'{name} alpha 탐색 결과'); ax.legend(); ax.grid(True)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'pancreas_optuna_search.png'), dpi=100)
plt.show()
print(f"탐색 결과 이미지 저장: {os.path.join(RESULTS_DIR, 'pancreas_optuna_search.png')}")


In [ ]:
# ── 최적 alpha 불러오기 (이전 셀이 실행됐다면 메모리에 있음,
#    없으면 JSON에서 로드) ────────────────────────────────────────────────────
try:
    _ = best_alpha_plwce
except NameError:
    with open(os.path.join(RESULTS_DIR, 'pancreas_optuna_results.json')) as f:
        d = json.load(f)
    best_alpha_plwce = d['plwce']['best_alpha']
    best_alpha_pwce  = d['pwce']['best_alpha']

FINAL_EPOCHS = 150   # 최종 비교 학습 epoch 수
FINAL_LR     = 1e-4

# ── 3가지 Loss 최종 학습 ─────────────────────────────────────────────────────
experiments = [
    ('ce_dice',   1.0,              'CE+Dice (기준선)'),
    ('plwce_dice', best_alpha_plwce, f'PLWCE+Dice (alpha={best_alpha_plwce:.2f})'),
    ('pwce_dice',  best_alpha_pwce,  f'PWCE+Dice  (alpha={best_alpha_pwce:.2f})'),
]

final_results = {}
for loss_name, alpha, label in experiments:
    print(f"\n>>> {label}")
    model, history, best_dice = train_transunet(
        loss_name = loss_name,
        alpha     = alpha,
        epochs    = FINAL_EPOCHS,
        lr        = FINAL_LR,
        subset_ratio = 1.0,
        tag       = 'final'
    )
    final_results[label] = {'model': model, 'history': history, 'best_dice': best_dice,
                            'loss_name': loss_name, 'alpha': alpha}

print("\n\n[최종 학습 결과 요약]")
print(f"{'실험':<35} {'Best Val mDice':>14}")
print("-" * 51)
for label, r in final_results.items():
    print(f"{label:<35} {r['best_dice']:>14.4f}")


In [ ]:
# ── Test set 평가 (클래스별 Dice) ────────────────────────────────────────────
def evaluate_on_testset(model):
    """Test volume(.h5)에 대해 클래스별 Dice 계산"""
    model.eval()
    dice_all = np.zeros(NUM_CLASSES - 1)  # fg only
    n_cases  = 0

    for h5_path in tqdm(test_files, desc="Test eval"):
        with h5py.File(h5_path, 'r') as f:
            vol   = f['image'][:]   # (D, H, W) or (H, W, D)
            label = f['label'][:]

        # 슬라이스 단위로 예측
        preds = np.zeros_like(label)
        for s in range(vol.shape[0]):
            sl = vol[s].astype(np.float32)
            sl = (sl - sl.min()) / (sl.max() - sl.min() + 1e-8)
            sl = cv2.resize(sl, (224, 224), interpolation=cv2.INTER_LINEAR)
            sl = torch.from_numpy(np.stack([sl]*3)).unsqueeze(0).float().to(device)
            with torch.no_grad():
                logit = model(sl)
                pred  = torch.argmax(logit, dim=1).squeeze().cpu().numpy()
            pred_orig = cv2.resize(pred.astype(np.float32),
                                   (label.shape[2], label.shape[1]),
                                   interpolation=cv2.INTER_NEAREST).astype(np.int64)
            preds[s] = pred_orig

        for c_idx, c in enumerate(range(1, NUM_CLASSES)):
            p = (preds  == c).astype(float)
            t = (label  == c).astype(float)
            inter = (p * t).sum()
            union = p.sum() + t.sum()
            if union > 0:
                dice_all[c_idx] += 2. * inter / (union + 1e-8)
        n_cases += 1

    return dice_all / n_cases

# ── 모든 모델 Test 평가 ───────────────────────────────────────────────────────
test_scores = {}
if test_files:
    for label, r in final_results.items():
        print(f"\n{label} 테스트 평가 중...")
        per_class_dice = evaluate_on_testset(r['model'])
        mdice = float(np.mean(per_class_dice))
        test_scores[label] = {'per_class': per_class_dice.tolist(), 'mDice': mdice}
        print(f"  mDice = {mdice:.4f}")
        for cn, dc in zip(CLASS_NAMES[1:], per_class_dice):
            print(f"    {cn:<15}: {dc:.4f}")
else:
    print("테스트 데이터 없음 — Val 결과로 대체")
    for label, r in final_results.items():
        dice_pc, mdice = compute_val_metrics(r['model'], val_loader)
        test_scores[label] = {'per_class': dice_pc.tolist(), 'mDice': mdice}

# ── 클래스별 Dice 비교 바차트 ─────────────────────────────────────────────────
fg_names = CLASS_NAMES[1:]
x = np.arange(len(fg_names))
width = 0.25
fig, ax = plt.subplots(figsize=(16, 6))
colors = ['#4878D0', '#EE854A', '#6ACC65']
for i, (label, scores) in enumerate(test_scores.items()):
    ax.bar(x + i * width, scores['per_class'], width,
           label=f"{label}  (mDice={scores['mDice']:.4f})", color=colors[i], alpha=0.85)
ax.set_xticks(x + width)
ax.set_xticklabels(fg_names, rotation=30, ha='right')
ax.set_ylabel('Dice Score'); ax.set_title('TransUNet — 클래스별 Dice 비교 (CE vs PLWCE vs PWCE)')
ax.legend(); ax.grid(axis='y', alpha=0.5); ax.set_ylim(0, 1.0)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'pancreas_class_dice.png'), dpi=100)
plt.show()

# ── 학습 곡선 ─────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for i, (label, r) in enumerate(final_results.items()):
    ax1.plot(r['history']['loss'],     label=label, color=colors[i])
    ax2.plot(r['history']['val_mdice'],label=label, color=colors[i])
ax1.set_title('Train Loss');   ax1.set_xlabel('Epoch'); ax1.legend(); ax1.grid(True)
ax2.set_title('Val mDice');    ax2.set_xlabel('Epoch'); ax2.legend(); ax2.grid(True)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'pancreas_training_curves.png'), dpi=100)
plt.show()

# ── 최종 결과 저장 ────────────────────────────────────────────────────────────
save = {}
for label, scores in test_scores.items():
    r = final_results[label]
    save[label] = {
        'loss_name'   : r['loss_name'],
        'alpha'       : r['alpha'],
        'best_val_mdice': r['best_dice'],
        'test_mDice'  : scores['mDice'],
        'per_class_dice': {cn: float(d) for cn, d in zip(fg_names, scores['per_class'])},
        'best_alpha_optuna': {
            'plwce': best_alpha_plwce,
            'pwce' : best_alpha_pwce,
        }
    }

with open(os.path.join(RESULTS_DIR, 'pancreas_final_results.json'), 'w', encoding='utf-8') as f:
    json.dump(save, f, indent=2, ensure_ascii=False)
print(f"\n최종 결과 저장: {os.path.join(RESULTS_DIR, 'pancreas_final_results.json')}")
print("\n[최종 결과 요약]")
print(f"{'실험':<40} {'Test mDice':>10}")
print("-" * 52)
for label, s in save.items():
    print(f"{label:<40} {s['test_mDice']:>10.4f}")
